# Vectorless RAG Demo

This notebook demonstrates how to use the Vectorless RAG system to index a small collection of documents and run queries using BM25, TF-IDF, and a hybrid approach.

In [ ]:
from pathlib import Path
from vectorless_rag import VectorlessRAG

# If running this notebook outside the repo after cloning, install editable package first:
# %pip install -e .


## Load example documents
We will use the small text files under `examples/data/`.

In [ ]:
data_dir = Path('examples/data')
docs = []
for p in sorted(data_dir.glob('*.txt')):
    docs.append({"id": p.stem, "title": p.name, "text": p.read_text(encoding='utf-8', errors='ignore')})
len(docs), [d['title'] for d in docs]

## Initialize and index
Instantiate `VectorlessRAG` and index the documents.

In [ ]:
rag = VectorlessRAG()
rag.index_documents(docs)

## Run a query (BM25)

In [ ]:
def show(results):
    for i, r in enumerate(results, 1):
        print(f"{i:02d}. [{r.get('method','?')}] score={r['score']:.3f}  {r.get('title') or r.get('id')}")

results = rag.retrieve('natural language', top_k=5, method='bm25')
show(results)

## Try TF-IDF and hybrid

In [ ]:
print('TF-IDF:')
show(rag.retrieve('natural language', top_k=5, method='tfidf'))

print('
Hybrid:')
show(rag.retrieve('natural language', top_k=5, method='hybrid'))

## Expansion and reranking
Toggle query expansion and heuristic reranking to see effect on scores.

In [ ]:
print('No expansion, no rerank:')
show(rag.retrieve('NLP', top_k=5, expand_query=False, rerank=False))

print('
With expansion, with rerank:')
show(rag.retrieve('NLP', top_k=5, expand_query=True, rerank=True))

## Statistics

In [ ]:
rag.get_statistics()